In [1]:
import os

# Include modules from path
import sys
sys.path.append(os.path.join("/home/mdafifal.mamun/research/S3M/src"))

: 

In [14]:
import matplotlib.pyplot as plt
import torch
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerModelCardData,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.evaluation import (
    EmbeddingSimilarityEvaluator,
    TripletEvaluator,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from sklearn.manifold import TSNE
from tqdm import tqdm

from data.buckets.bucket_data import BucketData, OtherBucketData
from data.buckets.issues_data import BucketDataset
from data.triplet_selector import RandomTripletSelector
from datasets import Dataset
from preprocess.entry_coders import Stack2Seq, Stack2SeqMultiStack
from preprocess.seq_coder import SeqCoder, SeqCoderMulti
from preprocess.tokenizers import SimpleTokenizer

In [2]:
# def format_stack(stack):
#     return stack


# def get_data_row(coder, event, similar_stack_id, dissimilar_stack_id):
#     return {
#         "anchor": format_stack(coder(event.st_id, transformer=True)),
#         "positive": format_stack(coder(similar_stack_id, transformer=True)),
#         "negative": format_stack(coder(dissimilar_stack_id, transformer=True)),
#         "ids": [event.st_id, similar_stack_id, dissimilar_stack_id],
#     }

In [3]:
from data.formatters import get_formatter
from 

In [15]:
triplet_selector_train = RandomTripletSelector(1)
triplet_selector_eval = RandomTripletSelector(1)
print("Load bucket data...")
bucket_data = OtherBucketData(
    "eclipse",
    "/home/mdafifal.mamun/research/S3M/dataset/EMSE_data/eclipse_2018/eclipse_stacktraces.json",
    3850,
    700,
    350,
    140,
)
bucket_data.load()
stack_loader = bucket_data.stack_loader()
data_gen = BucketDataset(bucket_data)
unsup_stacks = data_gen.train_stacks()

data_gen.reset()

stack2seq = Stack2SeqMultiStack(cased=False, trim_len=0, sep=".")

coder = SeqCoderMulti(
    stack_loader, stack2seq, SimpleTokenizer(), min_freq=0, max_len=None
)

# coder.fit(unsup_stacks)

Load bucket data...
Stack loader:  json_loader_multi


In [34]:
train_data = []
test_data = []
all_data = []
formatter = get_formatter("java")

print("Generate the dataset...")
for i, event in tqdm(enumerate(data_gen.train()), desc="Step"):
    similar_stack_ids, dissimilar_stack_ids = triplet_selector_train(event)
    for similar_stack_id, dissimilar_stack_id in zip(
        similar_stack_ids, dissimilar_stack_ids
    ):
        coded = coder(event.st_id, transformer=True)
        train_data.append(
            {
                "anchor": coder(event.st_id, transformer=True),
                "positive": coder(similar_stack_id, transformer=True),
                "negative": coder(dissimilar_stack_id, transformer=True),
                "ids": [event.st_id, similar_stack_id, dissimilar_stack_id],
            }
        )
        all_data.extend(coded)
        all_data.extend(coder(similar_stack_id, transformer=True))
        all_data.extend(coder(dissimilar_stack_id, transformer=True))

Generate the dataset...


Step: 0it [00:00, ?it/s]

Step: 5180it [01:05, 78.98it/s] 


In [44]:
frame_data = []

for frame_list in all_data:
    # Only keep unique frames
    frame_list = list(set(frame_list))
    frame_data.extend(frame_list)

In [26]:
from pprint import pprint

In [29]:
ctr = 0

for d in train_data:
    if len(d["negative"]) > 1:
        ctr+=1
        pprint(d["anchor"])
        print("----------------")
        pprint(d["positive"])
        break
print(ctr)
print("Frequency: ", ctr/len(train_data))

[['org.eclipse.core.launcher.main.main',
  'org.eclipse.core.launcher.main.run',
  'org.eclipse.core.launcher.main.basicrun',
  'java.lang.reflect.method.invoke',
  'org.eclipse.core.boot.bootloader.run',
  'org.eclipse.core.internal.boot.internalbootloader.run',
  'org.eclipse.core.internal.boot.internalbootloader.getrunnable',
  'java.lang.reflect.method.invoke',
  'org.eclipse.core.internal.runtime.internalplatform.loadergetrunnable',
  'org.eclipse.core.internal.plugins.configurationelement.createexecutableextension',
  'org.eclipse.core.internal.plugins.plugindescriptor.createexecutableextension',
  'org.eclipse.core.internal.plugins.plugindescriptor.createexecutableextension',
  'java.lang.class.newinstance',
  'java.lang.class.newinstance',
  'java.lang.classloader.loadclassinternal',
  'java.lang.classloader.loadclass',
  'org.eclipse.core.internal.boot.delegatingurlclassloader.loadclass',
  'org.eclipse.core.internal.boot.delegatingurlclassloader.loadclass',
  'org.eclipse.cor

In [45]:
from collections import Counter
import pandas as pd

# Step 1: Calculate Frequency of Each Frame
frame_counts = Counter(frame_data)

# Step 2: Convert to DataFrame for Easier Analysis
df = pd.DataFrame(frame_counts.items(), columns=["Frame", "Frequency"])

# Step 3: Add Percentage of Total
total_frames = len(train_data) * 3  # Total frames in the dataset
df["Percentage"] = (df["Frequency"] / total_frames) * 100

# Step 4: Sort by Frequency (Descending)
df = df.sort_values(by="Frequency", ascending=False).reset_index(drop=True)

# Step 5: Identify Common Frames (Threshold-Based)
# Adjust the threshold as needed (e.g., frames appearing in >20% of stack traces)
threshold = 20  # Percentage threshold for identifying common frames
common_frames = df[df["Percentage"] > threshold]

# Display Results
print("Top Frequent Frames:")
print(df.head(10))  # Top 10 frames by frequency

print("\nCommon Frames (Above Threshold):")
print(common_frames)

# Save Results to CSV for Further Inspection
df.to_csv("frame_analysis.csv", index=False)
common_frames.to_csv("common_frames.csv", index=False)

Top Frequent Frames:
                                               Frame  Frequency  Percentage
0                    java.lang.reflect.method.invoke      12172   82.466125
1    sun.reflect.delegatingmethodaccessorimpl.invoke      11323   76.714092
2        sun.reflect.nativemethodaccessorimpl.invoke      11255   76.253388
3    org.eclipse.swt.widgets.display.readanddispatch       9966   67.520325
4  org.eclipse.core.runtime.adaptor.eclipsestarte...       9593   64.993225
5     org.eclipse.ui.internal.workbench.runeventloop       9195   62.296748
6  org.eclipse.ui.internal.workbench.createandrun...       8455   57.283198
7    org.eclipse.ui.platformui.createandrunworkbench       8449   57.242547
8            org.eclipse.ui.internal.workbench.runui       8070   54.674797
9       org.eclipse.swt.widgets.eventtable.sendevent       7562   51.233062

Common Frames (Above Threshold):
                                                Frame  Frequency  Percentage
0                     java.lang.

In [82]:
frame_freq = {}

In [83]:
from pprint import pprint
# Find frame frequency from the training data
def x():
    
    for row in train_data:
        for stack in ["anchor", "positive", "negative"]:
            for frame in row[stack]:
                # if frame == "org.eclipse.swt.widgets.display.readanddispatch":
                #     pprint(row)
                #     return
                if frame not in frame_freq:
                    frame_freq[frame] = 0
                frame_freq[frame] += 1

x()

In [84]:
# Sort by frequency
frame_freq = dict(sorted(frame_freq.items(), key=lambda item: item[1], reverse=True))

In [85]:
# List frames that occurred more than 2000 times and save in a list
frames = []
for frame, freq in frame_freq.items():
    if freq > 1000:
        print(frame, freq)
        frames.append(frame)


java.awt.component.dispatchevent 10401
java.awt.container.dispatcheventimpl 10400
java.awt.eventqueue.dispatchevent 9394
java.awt.eventdispatchthread.pumpeventsforhierarchy 8513
java.awt.eventdispatchthread.pumpevents 8010
java.awt.eventdispatchthread.run 7970
org.netbeans.api.java.classpath.classpath.getclasspath 7733
org.netbeans.modules.web.freeform.webmodules.findclasspath 7683
org.netbeans.modules.java.project.projectclasspathprovider.findclasspath 7679
org.netbeans.modules.java.freeform.lookupmergerimplclasspathproviderimpl.findclasspath 7677
org.netbeans.modules.web.freeform.webmodulesffwebmodule.findclasspath 7674
java.awt.component.dispatcheventimpl 6192
java.awt.eventdispatchthread.pumponeeventforhierarchy 6175
org.openide.util.requestprocessorprocessor.run 5764
org.openide.util.requestprocessortask.run 5742
java.awt.window.dispatcheventimpl 5213
java.awt.component.processevent 5034
java.awt.container.processevent 5015
java.awt.lightweightdispatcher.dispatchevent 4224
java.aw

In [86]:
# From train data, remove all frequent frames
def remove_frequent_frames():
    for row in train_data:
        for stack in ["anchor", "positive", "negative"]:
            row[stack] = [frame for frame in row[stack] if frame not in frames]

In [87]:
frame_freq

{'java.awt.component.dispatchevent': 10401,
 'java.awt.container.dispatcheventimpl': 10400,
 'java.awt.eventqueue.dispatchevent': 9394,
 'java.awt.eventdispatchthread.pumpeventsforhierarchy': 8513,
 'java.awt.eventdispatchthread.pumpevents': 8010,
 'java.awt.eventdispatchthread.run': 7970,
 'org.netbeans.api.java.classpath.classpath.getclasspath': 7733,
 'org.netbeans.modules.web.freeform.webmodules.findclasspath': 7683,
 'org.netbeans.modules.java.project.projectclasspathprovider.findclasspath': 7679,
 'org.netbeans.modules.java.freeform.lookupmergerimplclasspathproviderimpl.findclasspath': 7677,
 'org.netbeans.modules.web.freeform.webmodulesffwebmodule.findclasspath': 7674,
 'java.awt.component.dispatcheventimpl': 6192,
 'java.awt.eventdispatchthread.pumponeeventforhierarchy': 6175,
 'org.openide.util.requestprocessorprocessor.run': 5764,
 'org.openide.util.requestprocessortask.run': 5742,
 'java.awt.window.dispatcheventimpl': 5213,
 'java.awt.component.processevent': 5034,
 'java.aw

In [88]:
from pprint import pprint
# Find frame frequency from the training data
def y():
    for row in train_data:
        for stack in ["anchor", "positive", "negative"]:
            for frame in row[stack]:
                if frame == "org.eclipse.swt.widgets.display.readanddispatch":
                    pprint(row)
                    return

y()

In [89]:
frame_freq["org.eclipse.ui.internal.workbench.run"]

KeyError: 'org.eclipse.ui.internal.workbench.run'

In [61]:
gg = ['org.eclipse.core.launcher.main.main',
              'org.eclipse.core.launcher.main.run',
              'org.eclipse.core.launcher.main.basicrun',
              'java.lang.reflect.method.invoke',
              'org.eclipse.core.boot.bootloader.run',
              'org.eclipse.core.internal.boot.internalbootloader.run',
              'org.eclipse.ui.internal.workbench.run',
              'org.eclipse.ui.internal.workbench.runeventloop',
              'org.eclipse.swt.widgets.display.readanddispatch',
              'org.eclipse.swt.widgets.display.rundeferredevents',
              'org.eclipse.swt.widgets.eventtable.sendevent',
              'org.eclipse.jface.action.actioncontributionitemactionlistener.handleevent',
              'org.eclipse.jface.action.actioncontributionitem.handlewidgetselection',
              'org.eclipse.ui.internal.wwinpluginaction.runwithevent',
              'org.eclipse.ui.internal.pluginaction.runwithevent',
              'org.eclipse.team.internal.ccvs.ui.actions.cvsaction.run',
              'org.eclipse.team.internal.ccvs.ui.actions.workspaceaction.beginexecution']

In [62]:
gg[-8:]

['org.eclipse.swt.widgets.display.rundeferredevents',
 'org.eclipse.swt.widgets.eventtable.sendevent',
 'org.eclipse.jface.action.actioncontributionitemactionlistener.handleevent',
 'org.eclipse.jface.action.actioncontributionitem.handlewidgetselection',
 'org.eclipse.ui.internal.wwinpluginaction.runwithevent',
 'org.eclipse.ui.internal.pluginaction.runwithevent',
 'org.eclipse.team.internal.ccvs.ui.actions.cvsaction.run',
 'org.eclipse.team.internal.ccvs.ui.actions.workspaceaction.beginexecution']

In [63]:
len(train_data)

4920

In [71]:
updated_train_data = []
updated_test_data = []

# Remove all frames from train_data that has a frequency over 2000
for row in train_data:
    new_row = {}
    for stack in ["anchor", "positive", "negative"]:
        previous = len(row[stack])

        if previous < 5:
            updated_train_data.append(row)
            continue
        
        new_row[stack] = [frame for frame in row[stack] if frame_freq[frame] <= 500]
        current = len(new_row[stack])
        print(previous, current)
        updated_train_data.append(new_row)


19 6
17 5
13 9
50 21
16 11
23 22
38 19
31 17
9 6
35 14
36 15
15 5
12 11
12 11
6 5
30 21
30 21
17 7
22 15
54 26
50 25
21 9
17 9
2 2
7 2
7 2
38 18
21 7
19 7
18 5
17 5
21 7
31 17
17 9
17 9
35 18
20 9
30 21
25 7
22 18
13 10
38 20
28 15
28 15
19 14
30 16
31 15
26 25
44 29
50 30
63 43
20 9
19 12
23 8
22 18
22 18
15 3
30 12
36 16
5 4
25 19
54 26
37 21
41 29
44 29
41 15
16 5
15 7
42 24
20 7
22 7
46 27
24 8
21 6
28 16
23 12
22 7
19 4
12 0
39 11
39 30
53 26
24 8
19 7
1 1
43 18
20 6
18 7
22 7
13 11
20 7
20 7
2 2
25 21
63 38
31 31
54 24
25 21
29 17
41 23
18 7
26 13
19 8
12 5
22 9
7 6
19 17
11 5
28 12
14 5
18 12
26 15
28 15
43 18
16 5
16 5
29 6
15 3
16 2
19 6
35 18
39 18
48 25
47 25
21 6
52 20
18 16
39 18
30 14
106 47
47 25
35 15
15 3
20 7
8 8
28 11
37 18
13 2
19 8
19 8
18 13
12 5
19 8
63 36
53 26
106 47
20 9
14 0
14 0
15 15
32 29
39 30
12 12
34 29
41 32
23 14
43 30
41 32
44 18
16 14
8 7
21 8
29 10
37 18
30 14
9 8
9 8
34 23
37 18
39 18
20 9
53 32
53 32
38 22
40 23
40 23
25 12
28 13
26 13
23 9
48 33

In [100]:
updated_train_data[-5]

{'anchor': [], 'positive': [], 'negative': []}

In [78]:
import pandas as pd
# Load json data into a pandas dataframe
df = pd.read_json("/home/mdafifal.mamun/research/S3M/dataset/EMSE_data/netbeans_2016/netbeans_stacktraces.json")

In [60]:
df.head()

,bug_id,bug_severity,bug_status,comments,component,creation_ts,delta_ts,description,dup_id,exception,keywords,op_sys,priority,product,rep_platform,resolution,short_desc,target_milestone,version,stacktrace
0,755,normal,CLOSED,[],Text,906750420,1230028497,External deleting .java file do not cloze open...,NaN,NaN,,All,P4,platform,All,DUPLICATE,External deleting .java file do not close open...,TBD,3.x,[{'exception': 'java.lang.ArrayIndexOutOfBound...
1,1106,normal,CLOSED,[],-- Other --,917921167,1230029124,This is the Windows Eastern European character...,NaN,NaN,,All,P2,platform,All,FIXED,RESOURCE_ENCODING set to Cp1250 in production ...,TBD,3.x,[{'exception': 'java.io.UnsupportedEncodingExc...
2,1329,normal,CLOSED,[],-- Other --,922132797,1230030864,Exception: /opt/local/jdk1.2/bin/java not foun...,NaN,NaN,,Other,P2,platform,All,FIXED,Execution does not work. In solaris (but maybe...,TBD,3.x,"[{'exception': 'java.io.IOException', 'message..."
3,1331,normal,CLOSED,[],-- Other --,922133018,1230031026,Exception occurred during event dispatching:\n...,NaN,NaN,,All,P2,platform,All,FIXED,"Menu ""Window"" sometimes can not be opened.",TBD,3.x,[{'exception': 'java.lang.NullPointerException...
4,1339,normal,CLOSED,[],Code,922135048,1027092528,[IAN] Problem of browser functionality in Swin...,NaN,NaN,,All,P4,javaee,All,FIXED,Very slow opening of pages with exception (in ...,TBD,3.x,[{'exception': 'javax.swing.text.StateInvarian...


# Analyze Relationship

In [75]:
from pprint import pprint

In [79]:
df_dup = df[df["dup_id"].notna()]

In [77]:
# Convert all dup_id to int
df["dup_id"] = df["dup_id"].apply(lambda x: int(x))

In [74]:
df.head()

,bug_id,bug_severity,bug_status,comments,component,creation_ts,delta_ts,description,dup_id,exception,keywords,op_sys,priority,product,rep_platform,resolution,short_desc,target_milestone,version,stacktrace
818,8811,normal,CLOSED,[],-- Other --,976205998,1229974614,[stable 3.1-22] On reinstaling module exceptio...,8766,NaN,,All,P3,platform,PC,DUPLICATE,ConCurrentModificationException in DataObjectPool,TBD,3.x,[{'exception': 'java.util.ConcurrentModificati...
848,9049,normal,CLOSED,[],-- Other --,979069728,1229981954,While renaming the name of a class and the nam...,8666,NaN,,Windows ME/2000,P1,platform,PC,FIXED,Invalid lock for file - FSException,3.x,3.x,[{'exception': 'org.openide.filesystems.FSExce...
894,9246,normal,CLOSED,[],Unsupported,980514610,1190798053,**WARNING: unexpected null reference in .class...,8117,NaN,,Linux,P4,java,PC,FIXED,NullPointerException at ParsingEnter.attribTre...,3.x,3.x,[{'exception': 'java.lang.NullPointerException...
910,9350,normal,CLOSED,[],Window System,981080000,1229968777,Obeying Exception dialog\n*********** Exceptio...,8868,NaN,,Other,P4,platform,Other,DUPLICATE,Exception when running an Gui app within the GUI,TBD,3.x,[{'exception': '2001java.lang.IllegalArgumentE...
958,9631,major,CLOSED,[],Window System,982196332,1229977917,"In my property editor, in setText, if the valu...",8413,NaN,,All,P3,platform,All,FIXED,Entering incorrect value into property sheet i...,3.x,3.x,"[{'exception': 'java.lang.Exception', 'message..."


In [217]:
def get_row(index):
    row = df_dup.iloc[index]
    # pprint(row)
    dup_id = int(row["dup_id"])
    main_st = df_dup.iloc[index]["stacktrace"][0]
    
    print("Bug id", row["bug_id"], "Dup id", dup_id)
    print("Main Desc", "="*20)
    pprint(df_dup.iloc[index]["description"])
    main_st = [st["function"] for st in main_st["frames"]]
    # pprint(main_st)
    dup_st = df[df["bug_id"] == dup_id]

    print("Dup Desc", "="*20)
    pprint(dup_st.iloc[0]["description"])

    dup_st = dup_st.iloc[0]["stacktrace"][0]
    # pprint(dup_st)
    dup_st = [st.get("function") for st in dup_st["frames"]]

    print("Main", "="*20)
    pprint(list(dict.fromkeys(main_st)))
    print("Dup", "="*20)
    pprint(list(dict.fromkeys(dup_st)))

    return list(dict.fromkeys(main_st)), list(dict.fromkeys(dup_st))

In [218]:
# Good 4, 22, 29, 30
main, dup = get_row(30)

Bug id 10614 Dup id 9794
Main Desc ====================
('I get the attached stack trace when creating a new JPanel\n'
 'using the wizard.  This occurs on the second page (Choose target)\n'
 'but only happens spuriously.\n'
 '\n'
 'Thu Mar 22 21:24:54 EST 2001java.lang.NullPointerException: null\n'
 'java.lang.NullPointerException\n'
 '        at ice.pilots.html4.ThePilot.clear(ice/pilots/html4/ThePilot)\n'
 '        at ice.storm.StormBase.$Gb(ice/storm/StormBase)\n'
 '        at ice.storm.StormBase.$Hb(ice/storm/StormBase)\n'
 '        at ice.storm.DefaultPilotContext.run(ice/storm/DefaultPilotContext)\n'
 '[catch] at java.lang.Thread.run(Thread.java:484)')
Dup Desc ====================
('If I select a package in the explorer and right-click ->New -> choose '
 'something\n'
 '(say Classes->Class), it tries to bring up the Template Chooser step instead '
 'of\n'
 'going directly to the name step and I get this exception.  If I dismiss it '
 'and\n'
 'do it again right away, it works.\n

In [41]:
# Sort by creation_ts
df = df.sort_values(by="creation_ts")
df.head()

,bug_id,bug_severity,bug_status,comments,component,creation_ts,delta_ts,description,dup_id,exception,keywords,op_sys,priority,product,rep_platform,resolution,short_desc,target_milestone,version,stacktrace
818,8811,normal,CLOSED,[],-- Other --,976205998,1229974614,[stable 3.1-22] On reinstaling module exceptio...,8766.0,NaN,,All,P3,platform,PC,DUPLICATE,ConCurrentModificationException in DataObjectPool,TBD,3.x,[{'exception': 'java.util.ConcurrentModificati...
848,9049,normal,CLOSED,[],-- Other --,979069728,1229981954,While renaming the name of a class and the nam...,8666.0,NaN,,Windows ME/2000,P1,platform,PC,FIXED,Invalid lock for file - FSException,3.x,3.x,[{'exception': 'org.openide.filesystems.FSExce...
894,9246,normal,CLOSED,[],Unsupported,980514610,1190798053,**WARNING: unexpected null reference in .class...,8117.0,NaN,,Linux,P4,java,PC,FIXED,NullPointerException at ParsingEnter.attribTre...,3.x,3.x,[{'exception': 'java.lang.NullPointerException...
910,9350,normal,CLOSED,[],Window System,981080000,1229968777,Obeying Exception dialog\n*********** Exceptio...,8868.0,NaN,,Other,P4,platform,Other,DUPLICATE,Exception when running an Gui app within the GUI,TBD,3.x,[{'exception': '2001java.lang.IllegalArgumentE...
958,9631,major,CLOSED,[],Window System,982196332,1229977917,"In my property editor, in setText, if the valu...",8413.0,NaN,,All,P3,platform,All,FIXED,Entering incorrect value into property sheet i...,3.x,3.x,"[{'exception': 'java.lang.Exception', 'message..."


In [151]:
from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer("/work/disa_lab/afif/projects/deduplication/trained_embedding_models/baai_bge-base-en-v1.5_netbeans_10frames_5train_0trim_4200_traindays/final")

/home/mdafifal.mamun/miniconda3/envs/tracesim/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [153]:
import os
os.getcwd()

'/home/mdafifal.mamun/research/S3M'

In [154]:
from src.data.formatters import get_formatter 

In [156]:
formatter = get_formatter("java", 10)

Selected formatter: java num_frames: 10


In [219]:
formatted_main = formatter.format(main)
formatted_dup = formatter.format(dup)
random_st, _ = get_row(1222)
formatted_random = formatter.format(random_st)

Bug id 42307 Dup id 36909
Main Desc ====================
("Here's the stack trace:\n"
 '\n'
 'Annotation: Exception occurred in Request Processor\n'
 'java.lang.IndexOutOfBoundsException: Index: 0\n'
 '\tat\n'
 'java.util.Collections$EmptyList.get(Collections.java:1628)\n'
 'at\n'
 'org.netbeans.modules.xml.text.completion.XMLCompletionQuery.query(XMLCompletionQuery.java:149)\n'
 'at\n'
 'org.netbeans.editor.ext.Completion.performQuery(Completion.java:581)\n'
 'at\n'
 'org.netbeans.editor.ext.Completion.access$700(Completion.java:47)\n'
 'at\n'
 'org.netbeans.editor.ext.Completion$1$QueryTask.run(Completion.java:511)\n'
 '\tat org.openide.util.Task.run(Task.java:136)\n'
 'at\n'
 'org.openide.util.RequestProcessor$Task.run(RequestProcessor.java:328)\n'
 '[catch] at\n'
 'org.openide.util.RequestProcessor$Processor.run(RequestProcessor.java:670)')
Dup Desc ====================
('[Nb build 2003101900, jdk1.4.2] \n'
 ' \n'
 'Steps to reproduce: \n'
 '--------------------------- \n'
 '1) ope

In [196]:
# FInd consine similarity with sentence transformer
from sklearn.metrics.pairwise import cosine_similarity

cosine_similarity(sbert_model.encode(formatted_main).reshape(1, -1), sbert_model.encode(formatted_dup).reshape(1, -1))

array([[0.8891839]], dtype=float32)

In [216]:
cosine_similarity(sbert_model.encode(formatted_main).reshape(1, -1), sbert_model.encode(formatted_random).reshape(1, -1))

array([[0.30909148]], dtype=float32)

In [15]:
ts = df.iloc[0]["creation_ts"]
# Convert timestamp to date
import datetime
start_date = datetime.datetime.fromtimestamp(ts)

In [16]:

date = datetime.datetime(2005, 12, 21)
diff = date - start_date
diff

datetime.timedelta(days=2643, seconds=39180)

In [20]:
# Add days to the date
new_date = start_date + datetime.timedelta(days=diff.days)

# List all rows that are within the date range
df[(df["creation_ts"] >= start_date.timestamp()) & (df["creation_ts"] <= new_date.timestamp())]

,bug_id,bug_severity,bug_status,comments,component,creation_ts,delta_ts,description,dup_id,exception,keywords,op_sys,priority,product,rep_platform,resolution,short_desc,target_milestone,version,stacktrace
0,755,normal,CLOSED,[],Text,906750420,1230028497,External deleting .java file do not cloze open...,NaN,NaN,,All,P4,platform,All,DUPLICATE,External deleting .java file do not close open...,TBD,3.x,[{'exception': 'java.lang.ArrayIndexOutOfBound...
1,1106,normal,CLOSED,[],-- Other --,917921167,1230029124,This is the Windows Eastern European character...,NaN,NaN,,All,P2,platform,All,FIXED,RESOURCE_ENCODING set to Cp1250 in production ...,TBD,3.x,[{'exception': 'java.io.UnsupportedEncodingExc...
2,1329,normal,CLOSED,[],-- Other --,922132797,1230030864,Exception: /opt/local/jdk1.2/bin/java not foun...,NaN,NaN,,Other,P2,platform,All,FIXED,Execution does not work. In solaris (but maybe...,TBD,3.x,"[{'exception': 'java.io.IOException', 'message..."
3,1331,normal,CLOSED,[],-- Other --,922133018,1230031026,Exception occurred during event dispatching:\n...,NaN,NaN,,All,P2,platform,All,FIXED,"Menu ""Window"" sometimes can not be opened.",TBD,3.x,[{'exception': 'java.lang.NullPointerException...
4,1339,normal,CLOSED,[],Code,922135048,1027092528,[IAN] Problem of browser functionality in Swin...,NaN,NaN,,All,P4,javaee,All,FIXED,Very slow opening of pages with exception (in ...,TBD,3.x,[{'exception': 'javax.swing.text.StateInvarian...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12962,70638,blocker,VERIFIED,[],Refactoring,1135087207,1175623334,"[release50-200512192030, JDK 1.5.0_06]\n\nStep...",70423.0,NaN,,All,P2,editor,All,DUPLICATE,IOE thrown when doing Refac. Push Down,5.x,5.x,[{'exception': 'javax.jmi.reflect.InvalidObjec...
12963,70640,blocker,RESOLVED,[],Code,1135089571,1135175738,Ran into this one during the attempt to delete...,NaN,NaN,,Windows XP,P3,guibuilder,PC,WORKSFORME,java.lang.NullPointerException at org.netbeans...,TBD,5.x,[{'exception': 'java.lang.NullPointerException...
12964,70644,blocker,VERIFIED,[],Autoupdate,1135092195,1161621560,Since the following commit it is impossible to...,NaN,NaN,,All,P1,platform,All,FIXED,Unable to connect to UC - stream closed,6.x,6.x,"[{'exception': 'java.io.IOException', 'message..."
12965,70650,blocker,VERIFIED,[],EJB,1135097572,1141651172,build 200512192030\n\nwhen creating session be...,NaN,NaN,,All,P2,javaee,Macintosh,FIXED,java.lang.reflect.UndeclaredThrowableException...,5.x,5.x,[{'exception': 'java.lang.reflect.UndeclaredTh...


In [3]:
df.iloc[0]["stacktrace"]

[{'exception': 'java.io.InterruptedIOException',
  'message': 'Read timed out',
  'frames': [{'function': 'java.net.SocketInputStream.socketRead',
    'file': 'NativeMethod',
    'fileline': None,
    'depth': 0},
   {'function': 'java.net.SocketInputStream.read',
    'file': 'SocketInputStream.java(CompiledCode',
    'fileline': None,
    'depth': 1},
   {'function': 'java.io.BufferedInputStream.fill',
    'file': 'BufferedInputStream.java(CompiledCode',
    'fileline': None,
    'depth': 2},
   {'function': 'java.io.BufferedInputStream.read',
    'file': 'BufferedInputStream.java(CompiledCode',
    'fileline': None,
    'depth': 3},
   {'function': 'org.eclipse.vcm.internal.core.ccvs.client.Connection.readLineOrUntil',
    'file': 'Connection.java(CompiledCode',
    'fileline': None,
    'depth': 4},
   {'function': 'org.eclipse.vcm.internal.core.ccvs.client.Connection.readToken',
    'file': 'Connection.java(CompiledCode',
    'fileline': None,
    'depth': 5},
   {'function': 'org.

In [68]:
updated_train_data

[{'anchor': ['org.eclipse.team.internal.ccvs.ui.actions.workspaceaction.beginexecution']},
 {'positive': ['org.eclipse.team.internal.ccvs.ui.actions.workspaceaction.beginexecution']},
 {'negative': ['com.oti.eclipsetools.internal.create.pluginimportwizard.run',
   'org.eclipse.debug.internal.core.launchmanager.resourcechanged',
   'org.eclipse.core.internal.events.resourcedelta.accept',
   'org.eclipse.debug.internal.core.launchmanagerlaunchmanagervisitor.visit']},
 {'anchor': ['org.eclipse.jdt.ui.actions.renameaction.run',
   'org.eclipse.jdt.internal.ui.refactoring.actions.renamejavaelementaction.run',
   'org.eclipse.jdt.internal.ui.refactoring.refactoringsupportfactoryrenamesupport.rename',
   'org.eclipse.jdt.internal.ui.refactoring.actions.performrefactoringutil.performrefactoring',
   'org.eclipse.jdt.internal.ui.refactoring.performchangeoperation.run',
   'org.eclipse.jdt.internal.ui.refactoring.performchangeoperation.executechange',
   'org.eclipse.jdt.internal.core.builder.ja

In [ ]:
frame_freq

{'org.eclipse.swt.widgets.display.readanddispatch': 8569,
 'java.lang.reflect.method.invoke': 8329,
 'sun.reflect.delegatingmethodaccessorimpl.invoke': 7803,
 'sun.reflect.nativemethodaccessorimpl.invoke': 7644,
 'org.eclipse.swt.widgets.eventtable.sendevent': 7112,
 'org.eclipse.swt.widgets.widget.sendevent': 6946,
 'org.eclipse.core.runtime.adaptor.eclipsestarter.run': 6201,
 'org.eclipse.ui.internal.workbench.runeventloop': 6006,
 'org.eclipse.ui.internal.workbench.createandrunworkbench': 5721,
 'org.eclipse.ui.platformui.createandrunworkbench': 5716,
 'org.eclipse.ui.internal.workbench.runui': 5474,
 'org.eclipse.swt.widgets.display.rundeferredevents': 4871,
 'org.eclipse.core.launcher.main.main': 4799,
 'org.eclipse.core.launcher.main.run': 4737,
 'org.eclipse.core.launcher.main.basicrun': 4732,
 'org.eclipse.ui.internal.workbench.run': 4363,
 'org.eclipse.core.runtime.platform.run': 3834,
 'org.eclipse.swt.custom.busyindicator.showwhile': 3696,
 'org.eclipse.core.runtime.internal

In [106]:
coder(6036, transformer=True)

[]

In [90]:
import matplotlib.pyplot as plt
import torch
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerModelCardData,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.evaluation import (
    EmbeddingSimilarityEvaluator,
    TripletEvaluator,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from sklearn.manifold import TSNE
from tqdm import tqdm

from data.buckets.bucket_data import BucketData, OtherBucketData
from data.buckets.issues_data import BucketDataset
from data.triplet_selector import RandomTripletSelector
from datasets import Dataset
from preprocess.entry_coders import Stack2Seq
from preprocess.seq_coder import SeqCoder
from preprocess.tokenizers import SimpleTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SentenceTransformer("BAAI/bge-base-en")
print(f"Model sequence length: {model.max_seq_length}")

# Find model size in CUDA
print(f"Model size: {torch.cuda.memory_allocated() / 1024 ** 3:.2f} GB")


dataset_key = "eclipse_pretrain_f"
run_name = "bge-base-eclipse_f"
generate_dataset = True
batch_size = 16
eval_size = 1000
num_frames = 10


def format_stack(stack):
    # Select last 10 frames
    stack = stack[-num_frames:]
    return "\n".join([f"{i+1}: {frame}" for i, frame in enumerate(stack)])


def get_data_row(coder, event, similar_stack_id, dissimilar_stack_id):
    return {
        "anchor": format_stack(coder(event.st_id, transformer=True)),
        "positive": format_stack(coder(similar_stack_id, transformer=True)),
        "negative": format_stack(coder(dissimilar_stack_id, transformer=True)),
    }


if generate_dataset:
    triplet_selector_train = RandomTripletSelector(1)
    triplet_selector_eval = RandomTripletSelector(1)
    print("Load bucket data...")
    bucket_data = OtherBucketData(
        "eclipse",
        "/home/mdafifal.mamun/research/S3M/dataset/EMSE_data/eclipse_2018/eclipse_stacktraces.json",
        3850,
        700,
        350,
        140,
    )
    bucket_data.load()
    stack_loader = bucket_data.stack_loader()
    data_gen = BucketDataset(bucket_data)
    unsup_stacks = data_gen.train_stacks()

    data_gen.reset()

    stack2seq = Stack2Seq(cased=False, trim_len=0, sep=".")

    coder = SeqCoder(
        stack_loader, stack2seq, SimpleTokenizer(), min_freq=0, max_len=None
    )

    coder.fit(unsup_stacks)

    train_data = []
    test_data = []

    print("Generate the dataset...")
    for i, event in tqdm(enumerate(data_gen.train()), desc="Step"):
        similar_stack_ids, dissimilar_stack_ids = triplet_selector_train(event)
        for similar_stack_id, dissimilar_stack_id in zip(
            similar_stack_ids, dissimilar_stack_ids
        ):
            train_data.append(
                get_data_row(coder, event, similar_stack_id, dissimilar_stack_id)
            )


    for i, event in tqdm(enumerate(data_gen.test()), desc="Step"):
        similar_stack_ids, dissimilar_stack_ids = triplet_selector_eval(event)
        for similar_stack_id, dissimilar_stack_id in zip(
            similar_stack_ids, dissimilar_stack_ids
        ):
            test_data.append(
                get_data_row(coder, event, similar_stack_id, dissimilar_stack_id)
            )


    frame_freq = {}
    for row in train_data:
        for stack in ["anchor", "positive", "negative"]:
            for frame in row[stack]:
                # if frame == "org.eclipse.swt.widgets.display.readanddispatch":
                #     pprint(row)
                #     return
                if frame not in frame_freq:
                    frame_freq[frame] = 0
                frame_freq[frame] += 1

    frame_freq = dict(
        sorted(frame_freq.items(), key=lambda item: item[1], reverse=True)
    )

    updated_train_data = []
    updated_test_data = []

    # Remove all frames from train_data that has a frequency over 2000
    for row in train_data:
        new_row = {}
        for stack in ["anchor", "positive", "negative"]:
            previous = len(row[stack])

            if previous < 5:
                updated_train_data.append(row)
                continue

            new_row[stack] = [frame for frame in row[stack] if frame_freq[frame] <= 500]
            current = len(new_row[stack])
            updated_train_data.append(new_row)


    # Remove all frames from test_data that has a frequency over 2000
    for row in test_data:
        new_row = {}
        for stack in ["anchor", "positive", "negative"]:
            previous = len(row[stack])

            if previous < 5:
                updated_test_data.append(row)
                continue

            new_row[stack] = [frame for frame in row[stack] if frame_freq[frame] <= 500]
            current = len(new_row[stack])
            updated_test_data.append(new_row)


    # Convert train_data list to a Dataset
    train_dataset = Dataset.from_list(updated_train_data)
    test_dataset = Dataset.from_list(updated_test_data)
    train_dataset.save_to_disk(f"datasets/{dataset_key}_train")
    test_dataset.save_to_disk(f"datasets/{dataset_key}_eval")

Using device: cuda
Model sequence length: 512
Model size: 0.41 GB
Load bucket data...
Generate the dataset...


Step: 5180it [01:06, 77.44it/s] 
Step: 2146it [00:17, 120.77it/s]


Saving the dataset (0/1 shards):   0%|          | 0/14760 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/6171 [00:00<?, ? examples/s]

In [97]:
frame_freq

{'e': 1018232,
 '.': 810441,
 'r': 643854,
 'i': 559653,
 't': 558910,
 'o': 498215,
 'n': 463346,
 'a': 458664,
 's': 444151,
 'c': 418155,
 'l': 394427,
 'p': 298665,
 'g': 248864,
 'd': 216169,
 'u': 181804,
 'm': 141581,
 ':': 135427,
 ' ': 135427,
 '\n': 120667,
 'w': 93692,
 'v': 82477,
 'f': 80700,
 'b': 72926,
 'h': 71929,
 'j': 64209,
 'y': 48722,
 'x': 38464,
 'k': 37940,
 '1': 26885,
 '2': 14526,
 '3': 14313,
 '4': 14045,
 '5': 13763,
 '6': 13459,
 '7': 13135,
 '8': 12836,
 '9': 12465,
 '0': 12125,
 'z': 11714,
 'q': 5411,
 '_': 1785}

In [96]:
for row in train_data:
    new_row = {}
    for stack in ["anchor", "positive", "negative"]:
        previous = len(row[stack])

        if previous < 5:
            updated_train_data.append(row)
            continue

        new_row[stack] = [frame for frame in row[stack] if frame_freq[frame] <= 500]
        print([frame for frame in row[stack] if frame_freq[frame] <= 500])
        current = len(new_row[stack])
        updated_train_data.append(new_row)
    break

[]
[]
[]


In [91]:
print("Load the preprocessed dataset")
train_dataset = Dataset.load_from_disk(f"datasets/{dataset_key}_train")
test_dataset = Dataset.load_from_disk(f"datasets/{dataset_key}_eval")

# Optionally preprocess and shuffle the dataset
train_dataset = train_dataset.shuffle(seed=42)
test_dataset = test_dataset.shuffle(seed=42)
eval_dataset = test_dataset.select(range(eval_size))

print("Dataset Sizes - Train:", len(train_dataset), "Test:", len(test_dataset))

# 4. Define a loss function
loss = MultipleNegativesRankingLoss(model)


# Populate the lists with the sentences and their labels from test_dataset
sentences_1 = []
sentences_2 = []
labels = []

for i in range(len(eval_dataset)):
    sentences_1.append(eval_dataset[i]["anchor"])
    sentences_2.append(eval_dataset[i]["positive"])
    labels.append(1)  # 1 means similar pair

    sentences_1.append(eval_dataset[i]["anchor"])
    sentences_2.append(eval_dataset[i]["negative"])
    labels.append(0)  # 0 means dissimilar pair


Load the preprocessed dataset
Dataset Sizes - Train: 14760 Test: 6171


In [92]:
sentences_1

[[],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],


In [98]:
test_dataset = Dataset.load_from_disk(f"datasets/eclipse_pretrain_f_eval")

In [99]:
test_dataset

Dataset({
    features: ['anchor', 'positive'],
    num_rows: 6171
})